# Direct Preference Optimization (DPO)

**Direct Preference Optimization (DPO)** aligns a language model with *human preferences* — but unlike classic RLHF it needs **no separate reward model and no reinforcement-learning loop**, which makes it much simpler and more stable to train.

## The setup

DPO learns from **preference pairs**. For a single prompt you provide two candidate answers:

- a **chosen** response (the one humans prefer), and
- a **rejected** response (the worse one).

DPO then nudges the model so it becomes **relatively more likely** to produce the *chosen* response and **less likely** to produce the *rejected* one.

## How it works (intuition)

DPO keeps two copies of the model:

- the **policy** model — the one we are training, and
- a frozen **reference** model — a snapshot of the starting model.

For every pair it looks at how the policy's probabilities for *chosen* vs. *rejected* have shifted **relative to the reference**. A single logistic (sigmoid) loss increases the margin

```
log π_policy(chosen) - log π_policy(rejected)
```

beyond what the reference assigns, while the reference term acts as a *leash* that stops the model from drifting so far it degenerates. A temperature-like hyper-parameter **`beta`** controls how tight that leash is (smaller `beta` lets the policy move further from the reference).

| | RLHF (PPO) | DPO |
|---|---|---|
| Separate reward model | Required (trained first) | **Not needed** |
| RL loop | Yes (sample → score → update) | **No** — one supervised-style loss |
| How data is used | Pairs → reward model → RL | Preference pairs used **directly** |
| Complexity / stability | Harder to tune | Simpler, more stable |

## What this notebook does

We fine-tune `Qwen/Qwen2.5-1.5B-Instruct` to sound more **human / natural** (chosen) and less like a stiff "I am an AI language model…" reply (rejected):

1. Load the instruct model in **8-bit**.
2. Load a **preference dataset** (prompt / chosen / rejected).
3. Format everything with Qwen's **chat template**.
4. Add **LoRA** adapters and train with **`DPOTrainer`**.
5. **Merge** the adapters and push the result to the Hub.

> **Requirements:** a GPU runtime and a Hugging Face token (loaded from `.env`). This notebook is not meant to run here — the outputs below were captured from a real GPU run.

## 1. Setup & authentication

First we install the libraries and sign in to the Hugging Face Hub.

- **`transformers`** provides the model and tokenizer, **`datasets`** loads the preference data, **`bitsandbytes`** provides 8-bit quantization, **`peft`** provides LoRA, and **`trl`** provides the `DPOTrainer` / `DPOConfig` used to run DPO.
- Authentication is needed to **download** the model and to **push** our fine-tuned result back to the Hub. The token is read from a local `.env` file (`HF_TOKEN=...`) instead of being hard-coded, so it never ends up in the notebook or in git history.

In [ ]:
# --- Install dependencies ---
# transformers: model + tokenizer   | datasets: load the preference data
# bitsandbytes: 8-bit quantization  | trl: DPOTrainer / DPOConfig
# peft: LoRA adapters                | huggingface_hub: login + push
# python-dotenv: read the HF token from a local .env file
# (-q = quiet output, -U = upgrade to the latest versions)
!pip install -q -U transformers datasets bitsandbytes  trl peft  huggingface_hub python-dotenv

In [ ]:
# Authenticate with the Hugging Face Hub using a token from a local .env file.
import os                          # read environment variables
from dotenv import load_dotenv     # load variables from a .env file
from huggingface_hub import login  # authenticate with the Hub

# Read the .env file in the current directory and set its keys as env vars:
#   HF_TOKEN=hf_your_token_here
load_dotenv()

# Fetch the token (os.getenv returns None if it isn't set).
hf_token = os.getenv("HF_TOKEN")

# Fail early with a clear message if the token is missing or still the placeholder.
if not hf_token or hf_token == "your_hugging_face_token_here":
    raise ValueError(
        "HF_TOKEN not found. Create a .env file with HF_TOKEN=hf_your_token_here "
        "(get a token at https://huggingface.co/settings/tokens)."
    )

# Log in for this session so downloads/uploads are authenticated.
login(token=hf_token)
print("Successfully authenticated with the Hugging Face Hub.")

Successfully authenticated with the Hugging Face Hub.


## 2. How DPO pads its batches

DPO is different from ordinary fine-tuning because every training example is a **triplet**: one `prompt`, one `chosen` completion, and one `rejected` completion. To batch triplets of different lengths together they must be padded to equal length — and the *prompt* and the *completions* are padded from **opposite sides**:

- The **prompt is LEFT-padded**, so its real tokens sit flush against the completion that follows it (the model reads the prompt and then immediately continues from it).
- The **chosen / rejected completions are RIGHT-padded**, since they are the part the model is scored on and we want them aligned from their first token.

The cell below is a **comment-only illustration** of this layout (no code runs) so you can picture the tensor shapes before we build the trainer.

In [ ]:
# DPO batches contain three sequences per example. The prompt is LEFT-padded
# (so it ends right before the completion), while the chosen/rejected completions
# are RIGHT-padded:
#
#   Prompt:    [PAD, PAD, 1, 2]     (left-pad)
#   Chosen:    [3, 4, 5, PAD]       (right-pad)
#   Rejected:  [6, PAD, PAD, PAD]   (right-pad)


## 3. Download and load the model (8-bit)

Here we **download and load** `Qwen/Qwen2.5-1.5B-Instruct` in **8-bit** via `BitsAndBytesConfig(load_in_8bit=True)`. This is the **policy** model — the one DPO will train.

- 8-bit quantization roughly halves the memory versus 16-bit, so a 1.5B model fits comfortably on a small GPU and leaves room for the LoRA adapters and the reference model that DPO creates internally.
- `device_map="auto"` lets Accelerate place the layers on the GPU automatically, and `trust_remote_code=True` allows the model's custom code if it needs any.
- We also load the matching **tokenizer**. Remember it is *loaded, not trained*: it only encodes text and carries Qwen's chat template (the `<|im_start|>` / `<|im_end|>` markers we use later).

When we print this model in the next section you will see that its attention/MLP projections are now `Linear8bitLt` layers — the 8-bit versions of the usual `Linear` layers.

In [ ]:
# --- Load the policy model (Qwen2.5 1.5B Instruct) in 8-bit ---
# DPO fine-tunes an already-instruct model to prefer 'chosen' over 'rejected' replies.
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

# The checkpoint we fine-tune (Qwen 2.5, 1.5B params, instruct variant).
name_name = "Qwen/Qwen2.5-1.5B-Instruct"

# Tell transformers to load the weights in 8-bit precision via bitsandbytes.
config_8bit = BitsAndBytesConfig(load_in_8bit=True)

# Download + instantiate the model -> this is the trainable "policy" model.
model_8bit = AutoModelForCausalLM.from_pretrained(
    name_name,                        # which checkpoint to load
    quantization_config=config_8bit,  # load the weights in int8
    trust_remote_code=True,           # allow the model's custom code if any
    device_map="auto",                # let accelerate place layers on the GPU
)

# Load the matching tokenizer (loaded, NOT trained -- it only encodes text and
# carries Qwen's chat template).
tokenizer = AutoTokenizer.from_pretrained(name_name, trust_remote_code=True)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]


Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]


vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]


merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]


tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]


## 4. Load the preference dataset

DPO learns from **preference pairs**: for the same prompt, a **chosen** (preferred) response and a **rejected** (worse) one.

We use [`HumanLLMs/Human-Like-DPO-Dataset`](https://huggingface.co/datasets/HumanLLMs/Human-Like-DPO-Dataset) and take just the first **300 rows** for a quick demo. Each row has exactly three fields — `prompt`, `chosen`, `rejected`.

Inspecting a single example makes the goal obvious: the **chosen** reply is warm and human-like, while the **rejected** reply is the stiff *"I am an artificial intelligence language model..."* style we want to train the model *away* from.

In [ ]:
# --- Preference dataset ---
# Each row has a prompt plus a 'chosen' (preferred) and a 'rejected' response.
from datasets import load_dataset

# split="train[:300]" downloads only the first 300 rows for a quick demo.
dataset = load_dataset("HumanLLMs/Human-Like-DPO-Dataset", split="train[:300]")
dataset   # shows the features (prompt / chosen / rejected) and the row count

README.md:   0%|          | 0.00/1.69k [00:00<?, ?B/s]


data.json:   0%|          | 0.00/28.0M [00:00<?, ?B/s]


Generating train split:   0%|          | 0/10884 [00:00<?, ? examples/s]


Dataset({
    features: ['prompt', 'chosen', 'rejected'],
    num_rows: 300
})

In [ ]:
# Inspect one preference triplet: 'chosen' is warm/human-like, while 'rejected'
# is the stiff 'I am an AI...' style we want to train AWAY from.
dataset[0]   # the first {prompt, chosen, rejected} example

{'prompt': 'Oh, I just saw the best meme - have you seen it?',
 'chosen': "😂 Ah, no I haven't! I'm dying to know, what's the meme about? Is it a funny cat or a ridiculous situation? Spill the beans! 🤣",
 'rejected': "I'm an artificial intelligence language model, I don't have personal experiences or opinions. However, I can provide you with information on highly-rated and critically acclaimed films, as well as recommendations based on specific genres or themes. Would you like me to suggest some notable movies or discuss a particular genre of interest?"}

## 5. Prepare the dataset for DPO (apply the chat template)

`DPOTrainer` expects `prompt`, `chosen`, and `rejected` to be **plain strings** already formatted the way the model reads them. Since Qwen is an instruct model, that means wrapping everything in its **chat template** (`<|im_start|>role ... <|im_end|>`).

We build this up in a few small steps:

1. **Preview** the template on a single response to see the `<|im_start|>` / `<|im_end|>` markers Qwen uses.
2. **Turn each raw prompt** (a bare string) into a proper `user` message so the template can render it.
3. **Format every row**: render the prompt through `apply_chat_template`, and wrap both the `chosen` and `rejected` responses in an `assistant` turn using a small string template.

After this step the dataset's three columns are all correctly-formatted strings, ready to hand straight to the trainer.

In [ ]:
# Wrap a response as a single 'assistant' message to preview chat-template formatting.
example_text = [{'role': 'assistant', 'content': dataset[0]['chosen']}]
example_text

[{'role': 'assistant',
  'content': "😂 Ah, no I haven't! I'm dying to know, what's the meme about? Is it a funny cat or a ridiculous situation? Spill the beans! 🤣"}]

In [ ]:
# Qwen's chat template uses <|im_start|> / <|im_end|> role markers.
# tokenize=False -> return the formatted STRING (not token ids) so we can read it.
tokenizer.apply_chat_template(example_text, tokenize=False)

"<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>assistant\n😂 Ah, no I haven't! I'm dying to know, what's the meme about? Is it a funny cat or a ridiculous situation? Spill the beans! 🤣<|im_end|>\n"

In [ ]:
# Turn each raw prompt string into a proper 'user' message (a list with one dict),
# which is the structure the chat template expects.
def apply_chat_temp(example):
    example['prompt'] = [{'role': 'user', 'content': example['prompt']}]
    return example

# .map applies the function to every row in the dataset.
new_dataset = dataset.map(apply_chat_temp)
new_dataset[0]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]


{'prompt': [{'content': 'Oh, I just saw the best meme - have you seen it?',
   'role': 'user'}],
 'chosen': "😂 Ah, no I haven't! I'm dying to know, what's the meme about? Is it a funny cat or a ridiculous situation? Spill the beans! 🤣",
 'rejected': "I'm an artificial intelligence language model, I don't have personal experiences or opinions. However, I can provide you with information on highly-rated and critically acclaimed films, as well as recommendations based on specific genres or themes. Would you like me to suggest some notable movies or discuss a particular genre of interest?"}

In [ ]:
# --- Format for DPO ---
# DPO expects 'prompt', 'chosen', 'rejected' as plain strings. Render the prompt via
# the chat template, and wrap both responses in an assistant turn using this template.
new_templete = """<|im_start|>assistant\n{model_answer}<|im_end|>\n"""

def format_dataset(example):
    # Render the user prompt (a messages list) into Qwen's template string.
    example['prompt'] = tokenizer.apply_chat_template(example['prompt'], tokenize=False)
    # Wrap the preferred answer as an assistant turn.
    example['chosen'] = new_templete.format(model_answer=example['chosen'])
    # Wrap the dispreferred answer the same way.
    example['rejected'] = new_templete.format(model_answer=example['rejected'])
    return example

formatted_dataset = new_dataset.map(format_dataset)
formatted_dataset[0]   # now prompt/chosen/rejected are all template strings

Map:   0%|          | 0/300 [00:00<?, ? examples/s]


{'prompt': '<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\nOh, I just saw the best meme - have you seen it?<|im_end|>\n',
 'chosen': "<|im_start|>assistant\n😂 Ah, no I haven't! I'm dying to know, what's the meme about? Is it a funny cat or a ridiculous situation? Spill the beans! 🤣<|im_end|>\n",
 'rejected': "<|im_start|>assistant\nI'm an artificial intelligence language model, I don't have personal experiences or opinions. However, I can provide you with information on highly-rated and critically acclaimed films, as well as recommendations based on specific genres or themes. Would you like me to suggest some notable movies or discuss a particular genre of interest?<|im_end|>\n"}

## 6. Add LoRA adapters to the model

Rather than fine-tuning all ~1.5B weights, we **freeze the base model** and inject small trainable **LoRA** matrices into each attention and MLP projection. Only these adapters are trained.

### The LoRA update

LoRA never modifies the original weight matrix `W`. It learns two small matrices `A` and `B` and adds their product as a **low-rank update**, scaled by `alpha / r`:

![LoRA update: W' = W + (alpha / r) * B * A](images/lora-formula.png)

- **`W`** — the original **frozen** pretrained weight matrix (shape `d x d`).
- **`A` and `B`** — the small **trainable** matrices: `A` has shape `r x d` and `B` has shape `d x r`. Their product `B*A` has the same shape as `W`, but is *low-rank* (rank `r`), so it stores far fewer numbers.
- **`W'`** — the effective weight used at run time. During training only `A` and `B` receive gradients; `W` stays fixed.

### The parameters we set in `LoraConfig`

| Parameter | Value here | What it does |
|-----------|-----------|--------------|
| `r` | `32` | **Rank** of the update — the inner dimension shared by `A` and `B`. Higher `r` = more capacity, but more trainable parameters. |
| `lora_alpha` | `32` | **Scaling factor `alpha`.** The update is multiplied by `alpha / r` before being added, controlling how strongly the adapters influence the output. |
| `target_modules` | `q/k/v/o`, `gate/up/down` | **Which layers get adapters** — the attention query/key/value/output projections and the MLP gate/up/down projections. |
| `lora_dropout` | `0.05` | **Dropout** on the LoRA path during training, for regularization. |
| `bias` | `"none"` | Whether bias terms are trained — here they are left frozen. |
| `task_type` | `"CAUSAL_LM"` | Tells PEFT this is a **causal language model** so it adapts the right layers. |

`get_peft_model(...)` then wraps the 8-bit model with these adapters, and `print_trainable_parameters()` confirms that only about **2.3%** (~37M of ~1.58B) of the parameters are trainable. We print the model **before** and **after** wrapping so you can see the projection layers gain `lora_A` / `lora_B` sub-modules while the original weights stay frozen.

In [ ]:
# The 8-bit Qwen model before LoRA (projections are Linear8bitLt layers).
model_8bit

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear8bitLt(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear8bitLt(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear8bitLt(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear8bitLt(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear8bitLt(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear8bitLt(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear8bitLt(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNor

In [ ]:
# --- Add LoRA adapters ---
# Freeze the base model and train only small rank-32 matrices (~2.3% of params).
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=32,                    # rank of the low-rank update (capacity)
    lora_alpha=32,           # scaling factor; the update is scaled by alpha/r
    target_modules=[         # which layers get adapters:
        "q_proj", "k_proj", "v_proj", "o_proj",   # attention projections
        "gate_proj", "up_proj", "down_proj",      # MLP projections
    ],
    lora_dropout=0.05,       # dropout on the LoRA path (regularization)
    bias="none",             # do not train bias terms
    task_type="CAUSAL_LM",   # task family: causal language modeling
)

# Wrap the model so forward passes add the LoRA update; base weights stay frozen.
lora_model = get_peft_model(model_8bit, lora_config)
lora_model.print_trainable_parameters()   # trainable vs total params

trainable params: 36,929,536 || all params: 1,580,643,840 || trainable%: 2.3364


In [ ]:
# The LoRA-wrapped model: each target projection now carries lora_A/lora_B adapters.
lora_model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear8bitLt(
                (base_layer): Linear8bitLt(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): 

## 7. Set the training arguments and run DPO training

Now we configure the **training arguments** and launch DPO.

**`DPOConfig`** holds the hyper-parameters:
- `beta` is the DPO-specific knob — the strength of the "leash" tying the policy to the frozen reference model. A smaller `beta` lets the policy move further from the reference; here it is a small `0.01`.
- The rest are standard: a batch of 2 with `gradient_accumulation_steps=4` (effective batch 8), a cosine-decayed learning rate, and a short `max_steps=40` demo run.

**`DPOTrainer`** then runs the training loop. A key detail: because we pass **no `ref_model`**, the trainer automatically makes a frozen copy of the current model *without the LoRA adapters* and uses it as the reference. It tokenizes the prompt/chosen/rejected columns, computes the DPO loss, and updates **only** the LoRA adapters. Watch the training loss fall in the output table; the final `TrainOutput` reports the average loss over the 40 steps.

### Three key training parameters explained

**1. `gradient_accumulation_steps`** — how many batches to process *before* updating the model weights. Instead of updating after every batch, the gradients are summed across several batches and applied once:

```
Batch 1 → gradients
Batch 2 → gradients
Batch 3 → gradients
Batch 4 → gradients
             ↓
       update weights
```

So `effective batch size = batch size × gradient_accumulation_steps` — here `2 × 4 = 8`. This is useful when your GPU cannot fit a large batch in memory.

**2. `learning_rate`** — the **step size** of each weight update ("how much should I change the model after learning from this batch?").

- Small LR → small updates → slower but more stable learning.
- Large LR → large updates → faster but can become unstable.

LoRA / DPO fine-tuning usually uses small values (e.g. `5e-6` to `3e-5`).

**3. `lr_scheduler_type`** — the learning rate normally does **not** stay constant during training. The scheduler decides *how it changes over time*; `"cosine"` decays it smoothly along a cosine curve from the starting value toward ~0.

In [ ]:
# --- DPO configuration ---
# DPOConfig extends the usual Hugging Face TrainingArguments with DPO-specific
# options. Every argument we set below is explained inline.
from trl import DPOTrainer, DPOConfig

dpo_config = DPOConfig(
    output_dir="./results",             # folder where checkpoints and logs are written
    per_device_train_batch_size=2,      # preference pairs processed per GPU on each step
    max_steps=40,                       # total optimizer steps to run (short demo); overrides epoch count
    gradient_accumulation_steps=4,      # sum grads over 4 steps before updating -> effective batch = 2 x 4 = 8
    learning_rate=3e-5,                 # optimizer step size (how large each weight update is)
    lr_scheduler_type="cosine",         # decay the learning rate along a cosine curve over training
    optim="adamw_torch",                # optimizer to use: PyTorch's AdamW
    logging_steps=10,                   # log the training metrics (loss, rewards, etc.) every 10 steps
    beta=0.01,                          # DPO strength: how tightly the policy is tied to the frozen reference
                                        #   model. Lower beta = looser leash = the policy may diverge more from
                                        #   the reference (a core DPO knob; typical values ~0.1, here a small 0.01).
    report_to="none",                   # disable external experiment loggers (Weights & Biases, TensorBoard, ...)
)

In [ ]:
# --- Train with DPO ---
# DPOTrainer raises the likelihood of 'chosen' over 'rejected'. Because no
# ref_model is passed, it internally copies the policy WITHOUT the LoRA adapters
# and freezes that copy to use as the reference model.
trainer = DPOTrainer(
    model=lora_model,                    # the trainable policy (LoRA-wrapped) model
    train_dataset=formatted_dataset,     # the prompt/chosen/rejected data
    # tokenizer=tokenizer,               # old argument name (deprecated)
    processing_class=tokenizer,          # new argument name for the tokenizer
    args=dpo_config,                     # the DPO config from the previous cell
)

trainer.train()   # run the DPO training loop (watch the loss fall)

<ipython-input-14-ea251014c22d>:1: FutureWarning: `tokenizer` is deprecated and removed starting from version 0.16.0 for `DPOTrainer.__init__`. Use `processing_class` instead.
  trainer = DPOTrainer(


Extracting prompt in train dataset:   0%|          | 0/300 [00:00<?, ? examples/s]


Applying chat template to train dataset:   0%|          | 0/300 [00:00<?, ? examples/s]


Tokenizing train dataset:   0%|          | 0/300 [00:00<?, ? examples/s]


No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Step,Training Loss
10,0.541000
20,0.141800
30,0.016600
40,0.010600


TrainOutput(global_step=40, training_loss=0.1774996966123581, metrics={'train_runtime': 199.7703, 'train_samples_per_second': 1.602, 'train_steps_per_second': 0.2, 'total_flos': 0.0, 'train_loss': 0.1774996966123581, 'epoch': 1.0533333333333332})

## 8. Merge the LoRA adapters and push to the Hub

Training only updated the small LoRA adapters. To get a **standalone** fine-tuned model we fold them back into the base weights:

1. **Save** the trained adapters to disk with `save_pretrained` — just a few MB.
2. **Reload the base model in full precision** (fp32). LoRA adapters *cannot* be merged into 8-bit weights, so we reload without a quantization config.
3. **Attach** the saved adapters to that fp32 model with `PeftModel.from_pretrained`.
4. **Merge** with `merge_and_unload()`, which computes `W <- W + (alpha/r)*B*A` for every adapted layer and then removes the LoRA wrappers — leaving plain `Linear` layers again.
5. **Push** the merged model and tokenizer to the Hub so they can be shared and reused (replace the repo id with your own namespace).

In [ ]:
# Save the trained LoRA adapters to disk so we can attach them to the fp32 model.
lora_path = "./lora_adapters"            # output directory for the adapter files
lora_model.save_pretrained(lora_path)    # writes just the small adapter weights

In [ ]:
# --- Reload the base model in full precision to merge the adapters into it ---
# LoRA adapters cannot be merged into 8-bit weights, so we reload in default (fp32).
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

lora_path = "./lora_adapters"            # where we saved the adapters above
name_name = "Qwen/Qwen2.5-1.5B-Instruct" # same checkpoint as before

# Reload WITHOUT a quantization_config -> full-precision weights.
base_model = AutoModelForCausalLM.from_pretrained(
    name_name,
    trust_remote_code=True,
    device_map="auto",
)

tokenizer = AutoTokenizer.from_pretrained(name_name, trust_remote_code=True)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


In [ ]:
# Attach the saved LoRA adapters onto the full-precision base model.
from peft import PeftModel

final_model = PeftModel.from_pretrained(base_model, lora_path)   # wrap fp32 model with adapters
final_model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(

In [ ]:
# Fold the adapters into the base weights and drop the LoRA wrappers, giving a
# standalone fine-tuned model (plain Linear layers again).
final_model = final_model.merge_and_unload()
final_model

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotary_emb): Qw

In [ ]:
# Push the merged model + tokenizer to the Hub (replace with your own repo id).
final_model.push_to_hub("Cagatayd/final_model")   # upload the model weights
tokenizer.push_to_hub("Cagatayd/final_model")     # upload the tokenizer